# VAW Phase 1 - Colab run: Stage B, scene layout (SegFormer) and key-pair depth (Depth Anything V2 Small)
**Use a GPU runtime** (Runtime -> Change runtime type -> T4). Needs notebook 05 (context features) done. Resumable. Expect roughly 20-40 minutes for 793 clips.
For every clip: where is the floor / road / walls / doors / vehicles, is it indoor or outdoor (with a confidence), how dark is it (a robust statistic, not the mean),
and, for the most active pair of people, who is nearer by depth, whether they are really on the same depth plane, and whether someone is pinned against a wall or vehicle.
This is the report's Algorithm 2 (depth-aware refinement) used for context only. Method and limits: `docs/ARCHITECTURE.md` section 10.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r requirements.txt
!python -m pytest -q

## 1. Run Stage B on every clip (model weights download on first use, about 120 MB)

In [ ]:
!python -m src.context.scene --config configs/colab.yaml

## 2. What did it find? (layout reliability is the thing to look at)
`reliable` means the segmentation was confident AND the camera is still. Dark or unusual scenes segment badly (a night alley came out as 'wall 68%, floor 31%'),
so unreliable layouts are ignored by the story step. Low light here means a dim image, not the time of day: night cameras in infrared mode look bright.

In [ ]:
import glob, json, collections
import numpy as np
R = '/content/drive/MyDrive/VAW_results/context'
place, rel, doors, low, agree = collections.Counter(), collections.Counter(), 0, collections.Counter(), []
files = sorted(glob.glob(f'{R}/*/*_layout.json'))
for f in files:
    L = json.load(open(f))
    place[L['place_type']] += 1
    rel[L['reliable']] += 1
    doors += len(L.get('doors', []))
    low[(L.get('light') or {}).get('low_light')] += 1
    try:
        a = json.load(open(f.replace('_layout', '_depth'))).get('depth_agree_frac')
        if a is not None:
            agree.append(a)
    except FileNotFoundError:
        pass
print(len(files), 'layouts | place type:', dict(place), '| reliable:', dict(rel), '| doors found:', doors, '| low light:', dict(low))
print('depth vs box-height agreement on decisive frames: mean', round(float(np.mean(agree)), 2) if agree else None, 'over', len(agree), 'clips')

## 3. Look at one layout (change CAT and CLIP)
Colours: walkable green, obstacle grey, door yellow, vehicle blue, sky light blue, vegetation dark green, furniture brown. Check it against the real frame.

In [ ]:
import numpy as np, cv2
from IPython.display import display
from PIL import Image
CAT, CLIP = 'Normal', 'Normal_v6'
labels = np.load(f'{R}/{CAT}/{CLIP}_layout.npz')['labels']
pal = np.array([[0,0,0],[60,180,60],[130,130,130],[255,230,0],[40,80,230],[170,210,255],[20,100,20],[150,100,60]], np.uint8)
display(Image.fromarray(cv2.resize(pal[labels], (384, 384), interpolation=cv2.INTER_NEAREST)))
print(json.load(open(f'{R}/{CAT}/{CLIP}_layout.json')))